# Complete Project Walkthrough
## Quantitative Explainable AI for Network Intrusion Detection

This notebook explains the complete CSE 498R project from the research problem to the final IEEE paper. It uses the project's **already measured outputs**, so most cells run quickly. Expensive experiment commands are included but disabled by default.

### What you will learn
1. What the IDS black-box problem means.
2. How CIC-IDS2017 and UNSW-NB15 were prepared without leakage.
3. How Random Forest and XGBoost were evaluated.
4. How TreeSHAP and LIME explanations were tested quantitatively.
5. How bootstrap intervals, Wilcoxon tests, Holm correction, and effect sizes support the conclusions.
6. How robustness audits and per-alert evidence mitigate decision opacity.
7. How all measured evidence was converted into the final IEEE-style paper.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

# Jupyter may start in the project folder, its parent, or another folder.
# Search the common locations instead of depending on Path.cwd() alone.
def find_project_root():
    current = Path.cwd().resolve()
    candidates = [current, current / '498R']
    for parent in current.parents:
        candidates.extend([parent, parent / '498R'])
    candidates.extend([
        Path.home() / 'Desktop' / '498R',
        Path(r'C:\Users\User\Desktop\498R'),
    ])

    checked = []
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if (candidate / 'config.yaml').is_file() and (candidate / 'outputs').is_dir():
            return candidate

    # Google Colab runs under /content and cannot access Windows files.
    # In Colab, request the small companion bundle and extract it.
    if Path('/content').is_dir():
        try:
            from google.colab import files
            import io
            import zipfile

            print('Google Colab detected.')
            print('Please upload: 498R_COLAB_BUNDLE.zip')
            uploaded = files.upload()
            bundle_names = [name for name in uploaded if name.lower().endswith('.zip')]
            if not bundle_names:
                raise FileNotFoundError('No ZIP bundle was uploaded.')

            destination = Path('/content').resolve()
            with zipfile.ZipFile(io.BytesIO(uploaded[bundle_names[0]])) as archive:
                for member in archive.infolist():
                    target = (destination / member.filename).resolve()
                    if destination not in target.parents and target != destination:
                        raise ValueError(f'Unsafe ZIP path: {member.filename}')
                archive.extractall(destination)

            for extracted in [destination / '498R', destination]:
                if (extracted / 'config.yaml').is_file() and (extracted / 'outputs').is_dir():
                    return extracted
            raise FileNotFoundError('The uploaded ZIP does not contain the expected 498R project data.')
        except ImportError:
            pass

    locations = '\n'.join(f'  - {path}' for path in checked)
    raise FileNotFoundError(
        'The 498R project folder could not be located. Checked:\n' + locations +
        '\nKeep this notebook inside the project, or upload 498R_COLAB_BUNDLE.zip in Colab.'
    )

ROOT = find_project_root()
OUTPUTS = ROOT / 'outputs'
EXTENDED = OUTPUTS / 'extended'
PAPER = ROOT / 'output' / 'pdf' / 'xai_ids_ieee_paper.pdf'

required_files = [
    ROOT / 'config.yaml',
    OUTPUTS / 'run_manifest.json',
    OUTPUTS / 'predictive_metrics.csv',
    OUTPUTS / 'explanation_summary.csv',
    EXTENDED / 'literature_purpose_coverage.csv',
]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError('Required project outputs are missing:\n  - ' + '\n  - '.join(missing))

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root found:', ROOT)
print('Working directory:', Path.cwd().resolve())
print('Required project files: OK')
print('Final paper exists:', PAPER.exists())

## 1. The research problem

A network intrusion detection system (IDS) classifies network flows as normal or malicious. Binary classification predicts **Normal versus Attack**; multiclass classification predicts the specific attack family.

Accurate machine-learning models can still be black boxes: they produce alerts without a defensible account of why a decision occurred. This project does not claim to eliminate the universal theoretical black-box problem. Instead, it solves a narrower operational problem:

> Convert opaque tree-model predictions into auditable, quantitatively tested, and rejectable per-alert evidence under declared interventions.

The project evaluates the classifier and explainer as one system using five dimensions: predictive quality, explanation fidelity, explanation stability, latency, and feature-count sensitivity.

## 2. End-to-end experimental workflow

```text
Raw CIC-IDS2017 and UNSW-NB15 files
        |
        v
Label audit, deduplication, and fixed train/test partitions
        |
        v
Training-only imputation and categorical encoding
        |
        v
Random Forest and XGBoost; binary and multiclass tasks; five seeds
        |
        v
Predictive metrics and per-class error analysis
        |
        v
TreeSHAP and LIME on the same 40 held-out flows/configuration
        |
        v
Fidelity controls, stability, latency, paired inference, ablations
        |
        v
Per-alert transparency reports and IEEE paper
```

In [ ]:
# Inspect the most important project files.
important = [
    'config.yaml', 'run_benchmark.py', 'run_extended_audit.py',
    'explain_alert.py', 'build_paper.py', 'verify_outputs.py',
    'src/xai_ids/benchmark.py', 'src/xai_ids/extended_audit.py',
    'src/xai_ids/transparency.py', 'src/xai_ids/paper.py',
    'PROJECT_HANDBOOK.md', 'LITERATURE_REQUIREMENTS_TRACEABILITY.md'
]
pd.DataFrame({
    'file': important,
    'exists': [(ROOT / p).exists() for p in important],
    'purpose': [
        'Experiment settings', 'Main benchmark entry point', 'Additional robustness audits',
        'Per-alert explanation entry point', 'Paper builder', 'Integrity checks',
        'Data, models, XAI, statistics', 'Literature-driven audits',
        'Auditable alert reports', 'IEEE PDF generator',
        'Complete written explanation', 'Requirements-to-evidence mapping'
    ]
})

## 3. Configuration and reproducibility

`config.yaml` is the experiment contract. It records dataset paths, model parameters, predictive seeds, explanation sample counts, LIME budget, fidelity deletion sizes, and feature-ablation sizes. The run manifest adds source-file hashes, row counts, software versions, and hardware information.

In [ ]:
import yaml

config = yaml.safe_load((ROOT / 'config.yaml').read_text(encoding='utf-8'))
manifest = json.loads((OUTPUTS / 'run_manifest.json').read_text(encoding='utf-8'))
display(config)
print('Predictive seeds:', config['predictive_seeds'])
print('Explanations per configuration:', config['evaluation']['explanation_samples'])
print('Recorded environment:', manifest['environment'])

## 4. Datasets and leakage prevention

### CIC-IDS2017
The public local preprocessing contains 2,830,743 flows and 28 selected features. To control computation while retaining minority labels, at most 25,000 rows per class are sampled, duplicates are removed, and a stratified 80/20 split is created. Because capture-day metadata is unavailable, this result is **not** presented as temporal generalization evidence.

### UNSW-NB15
The predefined 175,341-row training and 82,332-row test files are audited before modeling. The loader removes 78,519/32,361 exact duplicates and 2,255 test feature vectors also present in training, leaving 96,822 clean training rows and 47,716 clean test rows. The dataset owner does not describe this as a temporal holdout.

### Leakage controls
The `TabularPreprocessor` learns numeric medians, category dictionaries, category modes, and feature baselines from training data only. Unknown test categories receive an explicit unseen code. Labels and attack-category columns are excluded from the feature space.

In [ ]:
dataset_rows = []
for context, values in manifest['datasets'].items():
    dataset_rows.append({'context': context, **values})
display(pd.DataFrame(dataset_rows).fillna(''))

hash_rows = [
    {'path': path, 'bytes': meta['bytes'], 'sha256': meta['sha256']}
    for path, meta in manifest['files'].items()
]
display(pd.DataFrame(hash_rows))

## 5. Models and predictive evaluation

Two strong tabular model families are compared:

- **Random Forest:** 120 trees, depth 18, minimum leaf size 2, balanced bootstrap weights.
- **XGBoost:** 160 depth-7 trees, learning rate 0.08, row/column subsampling 0.85, balanced sample weights.

Every dataset-task-model combination is trained with seeds 42, 52, 62, 72, and 82, producing 40 predictive runs. Macro-F1 receives special attention because it gives rare attack classes equal weight.

In [ ]:
predictive = pd.read_csv(OUTPUTS / 'predictive_metrics.csv')
summary = (predictive.groupby(['dataset', 'task', 'model'])
           .agg(accuracy=('accuracy', 'mean'),
                balanced_accuracy=('balanced_accuracy', 'mean'),
                macro_f1=('macro_f1', 'mean'),
                macro_f1_sd=('macro_f1', 'std'),
                weighted_f1=('weighted_f1', 'mean'))
           .reset_index())
display(summary.round(4))

fig, ax = plt.subplots(figsize=(9, 4))
labels = summary['dataset'].str.replace('-IDS2017', '') + '/' + summary['task'] + '/' + summary['model'].str.replace('random_forest', 'RF').str.replace('xgboost', 'XGB')
ax.bar(labels, summary['macro_f1'], yerr=summary['macro_f1_sd'], capsize=3)
ax.set_ylabel('Macro-F1 (five-seed mean)')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=45)
ax.set_title('Predictive performance')
plt.tight_layout(); plt.show()

In [ ]:
# Correct attack-positive binary security metrics.
binary_security = (predictive[predictive['task'].eq('binary')]
                   .groupby(['dataset', 'model'])
                   [['false_positive_rate', 'false_negative_rate', 'pr_auc']]
                   .mean().reset_index())
display(binary_security.round(4))

per_class = pd.read_csv(OUTPUTS / 'per_class_metrics.csv')
rare_unsw = per_class[(per_class.dataset.eq('UNSW-NB15')) &
                      (per_class.task.eq('multiclass')) &
                      (per_class.seed.eq(42))]
display(rare_unsw.pivot(index='class', columns='model', values='recall').round(3))

### Interpretation

CIC binary macro-F1 is approximately 0.996-0.997 and CIC multiclass is approximately 0.925-0.932. After duplicate-leakage removal, UNSW binary is approximately 0.801-0.816 and UNSW multiclass is approximately 0.508-0.514. High weighted-F1 does not mean every attack family is detected well.

## 6. TreeSHAP and LIME

A feature attribution is a signed value describing how a feature influences one prediction. TreeSHAP directly uses the fitted tree structure. LIME perturbs one flow and fits a local surrogate.

The comparison is matched: both methods explain the same 40 label-stratified held-out flows for each of eight dataset-task-model configurations. Available binary TP, FP, TN, and FN cases are explicitly represented. Raw explanations are stored as signed vectors in the original feature order rather than being evaluated from plots.

In [ ]:
attributions = pd.read_csv(OUTPUTS / 'attributions.csv')
example = attributions[(attributions.dataset.eq('UNSW-NB15')) &
                       (attributions.task.eq('binary')) &
                       (attributions.model.eq('xgboost')) &
                       (attributions.explainer.eq('TreeSHAP')) &
                       (attributions.explanation_id.eq(0))].copy()
example['abs_attribution'] = example['attribution'].abs()
display(example.nlargest(10, 'abs_attribution')[
    ['actual_class', 'predicted_class', 'confidence', 'feature', 'attribution']
].round(5))

## 7. Fidelity: do the important features actually control the output?

For predicted class `c`, the project replaces the top `k` features with training-derived baseline values and measures the probability drop:

$$D_k(x,r)=p(c\mid x)-p(c\mid mask(x,r_{1:k},b))$$

The primary score averages `k = 1, 3, 5, 10`. The same operation is applied to random and bottom-ranked features. A ranking is convincing only if its top features outperform these controls.

In [ ]:
xai = pd.read_csv(OUTPUTS / 'explanation_summary.csv')
display(xai[[
    'dataset', 'task', 'model', 'explainer', 'fidelity_mean',
    'random_fidelity_mean', 'bottom_fidelity_mean',
    'fidelity_lift_vs_random', 'flip_rate'
]].round(4))

In [ ]:
controls = pd.read_csv(OUTPUTS / 'fidelity_control_tests.csv')
controls['passed'] = (controls['ci_low'] > 0) & (controls['holm_adjusted_p'] < 0.05)
display(controls.round(5))
print('Corrected controls passed:')
display(controls.groupby('explainer')['passed'].agg(['sum', 'count']))

### Main fidelity result

TreeSHAP passes all 8 corrected top-versus-random controls. LIME passes 7 of 8 and fails only for CIC binary Random Forest. This does not mean every failed explanation is useless; it means that ranking was not demonstrably better than random under the declared protocol.

## 8. Matched statistical comparison

TreeSHAP and LIME explain identical flows, so their fidelity scores are compared within each flow. The analysis uses:

- 2,000 paired bootstrap resamples for a 95% confidence interval.
- A one-sided Wilcoxon signed-rank test.
- Holm correction across eight comparisons.
- Paired standardized effect $d_z$.

A positive confidence interval shows consistent direction. The adjusted p-value controls false-positive risk. The raw difference and effect size describe magnitude.

In [ ]:
paired = pd.read_csv(OUTPUTS / 'paired_comparisons.csv')
display(paired.round(5))

labels = paired['dataset'].str.replace('-IDS2017', '') + '/' + paired['task'] + '/' + paired['model'].str.replace('random_forest', 'RF').str.replace('xgboost', 'XGB')
means = paired['mean_difference_shap_minus_lime']
xerr = np.vstack([means - paired['ci_low'], paired['ci_high'] - means])
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(means, np.arange(len(paired)), xerr=xerr, fmt='o', capsize=3)
ax.axvline(0, color='black', linestyle='--')
ax.set_yticks(np.arange(len(paired)), labels)
ax.set_xlabel('Mean fidelity difference: TreeSHAP - LIME')
ax.set_title('Paired fidelity differences with 95% bootstrap intervals')
plt.tight_layout(); plt.show()

All eight confidence intervals are above zero. Mean improvements range from about 0.062 to 0.223, paired effects from about 0.382 to 0.974, and the largest Holm-adjusted p-value is approximately 0.0032. TreeSHAP therefore has stronger primary deletion fidelity in every tested context.

## 9. Stability and latency

Neighborhood stability compares nearby held-out flows with the same predicted class. Signed cosine similarity measures attribution direction/magnitude agreement; top-five Jaccard measures ranking overlap. LIME is also repeated with five seeds on the same flows.

Latency is measured after warm-up and summarized with median, P90, P95, P99, and throughput. Stability is not fidelity: an explanation can be consistently wrong. Speed is also not fidelity: a fast unsupported explanation should not be deployed.

In [ ]:
display(xai[[
    'dataset', 'task', 'model', 'explainer',
    'stability_cosine_mean', 'stability_jaccard_mean',
    'self_stability_cosine_mean', 'latency_median_ms', 'latency_p95_ms'
]].round(4))

stability_dist = pd.read_csv(EXTENDED / 'stability_distributions.csv')
display(stability_dist.round(4))

TreeSHAP neighborhood cosine ranges from about 0.543 to 0.947, versus 0.472 to 0.801 for LIME. XGBoost TreeSHAP ranges from roughly 1.6-23.5 ms median explainer-call latency. UNSW Random Forest TreeSHAP is much slower and fits lower-volume or analyst-on-demand use.

## 10. Literature-driven robustness audits

The extended audit checks several conclusions that the main experiment alone cannot establish:

1. Majority and shallow-tree predictive baselines.
2. Class weighting versus no weighting.
3. Binary probability calibration using Brier score and expected calibration error (ECE).
4. Five masking references: median/mode, zero, marginal training row, nearest same prediction, and nearest opposite prediction.
5. LIME budgets of 100, 350, and 700 samples.
6. Fidelity by class and prediction error type.

In [ ]:
baselines = pd.read_csv(EXTENDED / 'predictive_baselines.csv')
balancing = pd.read_csv(EXTENDED / 'balancing_ablation.csv')
calibration = pd.read_csv(EXTENDED / 'calibration_summary.csv')
masking = pd.read_csv(EXTENDED / 'masking_baseline_ablation.csv')
lime_budget = pd.read_csv(EXTENDED / 'lime_budget_ablation.csv')

display(Markdown('### Simple baselines'))
display(baselines[['dataset', 'task', 'baseline', 'macro_f1', 'balanced_accuracy']].round(4))
display(Markdown('### Calibration'))
display(calibration.round(4))
display(Markdown('### Mean deletion by masking reference'))
display(masking.groupby('baseline')['deletion_score'].agg(['mean', 'median', 'min', 'max']).round(4))
display(Markdown('### LIME budget trade-off'))
display(lime_budget.groupby('lime_samples').agg(
    fidelity=('mean_deletion_fidelity', 'mean'),
    stability=('mean_self_stability_cosine', 'mean'),
    latency_ms=('median_latency_ms', 'median')).round(4))

### Why the reference audit matters

Pooled TreeSHAP deletion is about 0.392 with the primary training median/mode reference, 0.418 with zero, 0.386 with a marginal row, and 0.310 with the nearest opposite-prediction flow. It is about -0.004 with a nearest same-prediction flow. These references answer different questions. Fidelity is therefore a property of the model, input, explanation, masking rule, and reference distribution—not a universal causal truth.

In [ ]:
error_fidelity = pd.read_csv(EXTENDED / 'error_type_fidelity.csv')
class_fidelity = pd.read_csv(EXTENDED / 'class_specific_fidelity.csv')
display(Markdown('### Fidelity by error type'))
display(error_fidelity.round(4))
display(Markdown('### Example class-specific fidelity: UNSW multiclass'))
display(class_fidelity[(class_fidelity.dataset.eq('UNSW-NB15')) &
                       (class_fidelity.task.eq('multiclass'))].round(4))

## 11. Feature-count ablation

Global feature rankings are obtained from the full frozen model. New models are retrained with the top 5, 10, and 20 features and compared with the complete feature set. This tests whether a smaller input space preserves detection quality and reduces explanation cost. No universal feature count is assumed.

In [ ]:
ablation = pd.read_csv(OUTPUTS / 'feature_ablation.csv')
display(ablation[['dataset', 'task', 'model', 'feature_count', 'macro_f1', 'shap_latency_ms']].round(4))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, dataset in zip(axes, ['CIC-IDS2017', 'UNSW-NB15']):
    subset = ablation[ablation.dataset.eq(dataset)]
    for (task, model), group in subset.groupby(['task', 'model']):
        ax.plot(group.feature_count, group.macro_f1, marker='o', label=f'{task}/{model}')
    ax.set_title(dataset); ax.set_xlabel('Feature count'); ax.set_ylabel('Macro-F1'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 12. Per-alert black-box mitigation

`explain_alert.py` loads a saved model and produces Markdown and JSON containing:

- Predicted class and all class probabilities.
- Top signed TreeSHAP evidence and observed feature values.
- Alert-specific top, bottom, and 200-random-ranking deletion evidence.
- An empirical local p-value.
- Benchmark-level corrected evidence, stability, latency, and masking-reference sensitivity.
- A conservative tier: `SUPPORTED_FOR_THIS_ALERT`, `REVIEW_REQUIRED`, or `UNSUPPORTED`.

The report explicitly says that attribution is not causality and that support applies under the declared intervention.

In [ ]:
report_path = OUTPUTS / 'alert_reports' / 'unsw_binary_xgb_example.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('Evidence tier:', report['evidence_tier'])
print('Prediction:', report['alert']['predicted_class'])
print('Probability:', round(report['alert']['predicted_probability'], 6))
print('Local lift:', round(report['local_fidelity_evidence']['top_minus_random_lift'], 6))
print('Local empirical p:', round(report['local_fidelity_evidence']['local_empirical_one_sided_p'], 6))
display(pd.DataFrame(report['top_signed_features']))

## 13. Answers to the research questions

### RQ1 — Predictive performance
Both models outperform simple baselines. CIC detection is strong; performance on the cleaned predefined UNSW partition is substantially lower, and neither model solves its multiclass minority-class problem.

### RQ2 — Fidelity
TreeSHAP exceeds LIME in all eight primary matched comparisons and passes all random-ranking controls. Absolute deletion magnitude remains reference-dependent.

### RQ3 — Stability
TreeSHAP is more consistent for nearby same-prediction flows. LIME stability improves with a larger perturbation budget, at additional latency cost.

### RQ4 — Speed
No explainer has a universal speed advantage. XGBoost plus TreeSHAP is the strongest near-real-time candidate; UNSW Random Forest TreeSHAP is better suited to lower-volume investigation.

### RQ5 — Robustness
TreeSHAP's paired primary fidelity advantage persists across every tested context. Calibration, balancing, feature count, masking magnitude, and latency remain context-dependent.

## 14. How the IEEE paper was written

The benchmark was completed before the final claims were written. Every major paper claim comes from a machine-readable output:

- Predictive claims: `predictive_metrics.csv` and `per_class_metrics.csv`.
- Fidelity and latency: `explanation_summary.csv`.
- Paired superiority: `paired_comparisons.csv`.
- Random controls: `fidelity_control_tests.csv`.
- Feature reduction: `feature_ablation.csv`.
- Robustness qualifications: `outputs/extended/`.

`src/xai_ids/paper.py` reads these files, creates IEEE-style two-column tables and prose, and builds `output/pdf/xai_ids_ieee_paper.pdf`. The PDF was programmatically checked and visually inspected page by page.

In [ ]:
coverage = pd.read_csv(EXTENDED / 'literature_purpose_coverage.csv')
display(coverage)
print('Final paper:', PAPER)
print('Paper size (KB):', round(PAPER.stat().st_size / 1024, 1))

## 15. Reproduction commands

The main benchmark and extended audit are computationally expensive. Run them only when you intentionally want to regenerate the measured evidence.

In [ ]:
# SAFE DEFAULT: commands are printed, not executed.
commands = [
    r'.\.venv\Scripts\python.exe run_benchmark.py --config config.yaml',
    r'.\.venv\Scripts\python.exe run_extended_audit.py --config config.yaml',
    r'.\.venv\Scripts\python.exe build_paper.py',
    r'.\.venv\Scripts\python.exe -B verify_outputs.py',
    r'.\.venv\Scripts\python.exe -B -m unittest discover -s tests -v',
]
print('\n'.join(commands))

# To run one intentionally, remove the leading # from a line below.
# import subprocess
# subprocess.run(commands[0], cwd=ROOT, shell=True, check=True)

## 16. Final limitations and takeaway

The project measures predictive quality, fitted-model fidelity, stability, robustness, and operational cost. It does **not** prove real-world causality or that analysts make better decisions. A human-subject analyst study, drift evaluation, probability recalibration, conditional causal interventions, additional explainers, and adversarial testing remain future work.

The final defensible conclusion is:

> The framework mitigates the operational black-box problem for the tested tree-based IDS models by producing auditable, quantitatively validated, and rejectable per-alert evidence. TreeSHAP is the stronger tested explainer under the primary protocol, while intervention choice, model family, dataset shift, feature count, and latency remain essential qualifications.

For more detail, read `PROJECT_HANDBOOK.md`, `LITERATURE_REQUIREMENTS_TRACEABILITY.md`, and the final PDF.